# Analysis of Inverse Convolution Functions $G^{-1} * F$ and $F^{-1} * G$

In this notebook, we analyze the behavior of the convolution of inverses of the left and right augmented Chow functions ($G$ and $F$).

## Definitions

*   **Chow polynomial** ($H_M(x)$): The polynomial associated with the Chow ring of a matroid.
*   **Reverse Kazhdan-Lusztig polynomial** ($f^{\text{rev}}_M(x)$): $P_M^!(x) = x^{\text{rk} M} P_M(x^{-1})$.
*   **Right augmented Chow function** ($F$): $F = H * f^{\text{rev}}$.
*   **Left augmented Chow function** ($G$): $G = f^{\text{rev}} * H$.

The convolution $(A * B)(M)$ is defined as $\sum_{F \in \mathcal{L}(M)} A(M|F) B(M/F)$.

In [52]:
import sys
import os

# Setup paths
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

from sage.all import *
from custom_matroid_functions import *

P.<x> = LaurentPolynomialRing(ZZ)

def chow_poly_fn(M):
    return P(chow_polynomial(M))

def kl_rev_fn(M):
    p = kl_poly(M)
    t = p.parent().gen()
    rk = M.rank()
    return P(p.subs({t: x**-1}) * x**rk)

def get_matroid_key(M):
    rank = M.rank()
    size = len(M.groundset())
    try:
        can = M.canonical_form()
        return (rank, size, str(can))
    except AttributeError:
        bases = tuple(sorted(tuple(sorted(b)) for b in M.bases()))
        return (rank, size, str(bases))

def convolution_inverse(f):
    cache_inv = {}
    
    def f_inv(M):
        m_key = get_matroid_key(M)
        if m_key in cache_inv:
            return cache_inv[m_key]
        
        if M.rank() == 0 and M.size() == 0:
            res = 1 / f(M)
            cache_inv[m_key] = res
            return res
        
        # f_inv(M) = - (1/f(empty)) * sum_{empty != F <= E} f(M|F) f_inv(M/F)
        empty_matroid = matroids.Uniform(0, 0)
        f0 = f(empty_matroid)
        
        L = M.lattice_of_flats()
        E = M.groundset()
        
        total = P(0)
        for F in L:
            if len(F) == 0: continue
            restriction = M.delete(list(set(E) - set(F)))
            contraction = M.contract(list(F))
            total += f(restriction) * f_inv(contraction)
            
        res = -total / f0
        cache_inv[m_key] = res
        return res
    
    return f_inv

chow = chow_polynomial
chibar = reduced_characteristic_polynomial
char_poly = characteristic_polynomial
kl_poly = lambda M: M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)
grev = lambda M: P(x**M.rank())
mu_rev = lambda M: P(x**M.rank() * char_poly(M)(0))
mu = lambda M: P(char_poly(M)(0))
F_fn = convolve(chow_poly_fn, kl_rev_fn)
G_fn = convolve(kl_rev_fn, chow_poly_fn)
bbasepoly = convolve(mu_rev, F_fn)
Z_poly = convolve(grev, kl_poly)
q_poly = invkl
Y_poly = convolution_inverse(Z_poly)

F_inv_fn = convolution_inverse(F_fn)
G_inv_fn = convolution_inverse(G_fn)

G_inv_star_F = convolve(G_inv_fn, F_fn)
F_inv_star_G = convolve(F_inv_fn, G_fn)

F_inv_grev = convolve(F_inv_fn, grev)
G_inv_frev = convolve(G_inv_fn, kl_rev_fn)

## Analysis for Rank 3 Matroids

We test several rank 3 matroids to observe the pattern of $(G^{-1} * F)(M)$.

In [53]:
matroids_to_test = [
    (matroids.Uniform(0, 0), "U(0, 0)"),
    (matroids.Uniform(1, 1), "U(1, 1)"),
    (matroids.Uniform(2, 2), "U(2, 2)"),
    (matroids.Uniform(2, 3), "U(2, 3)"),
    (matroids.Uniform(3, 3), "U(3, 3)"),
    (matroids.Uniform(3, 4), "U(3, 4)"),
    (matroids.Uniform(3, 5), "U(3, 5)"),
    (matroids.Uniform(3, 6), "U(3, 6)"),
    (matroids.Uniform(4, 5), "U(4, 5)"),
    (matroids.CompleteGraphic(4), "K(4)"),
    (matroids.CompleteGraphic(5), "K(5)"),
    #(matroids.CompleteGraphic(6), "K(6)")
]

for M, label in matroids_to_test:
    print(f"{label}: chow = {chow(M)(x)}")
    print(f"{label}: bbasepoly = {bbasepoly(M)}")
    print(f"{label}: chibar = {chibar(M)(x)}")
    print(f"{label}: Z_poly = {Z_poly(M)}")
    print(f"{label}: q_poly = {q_poly(M)}")
    print(f"{label}: Y_poly = {Y_poly(M)}")
    print(f"{label}: F_fn = {F_fn(M)(x)}")
    print(f"{label}: F_inv_grev = {F_inv_grev(M)}")
    print(f"{label}: G_inv_frev = {G_inv_frev(M)}")
    print(f"{label}: G_inv_star_F = {G_inv_star_F(M)}")
    print(f"{label}: F_inv_star_G = {F_inv_star_G(M)}")
    print("*****************************\n")

U(0, 0): chow = 1
U(0, 0): bbasepoly = 1
U(0, 0): chibar = -1
U(0, 0): Z_poly = 1
U(0, 0): q_poly = 1
U(0, 0): Y_poly = 1
U(0, 0): F_fn = 1
U(0, 0): F_inv_grev = 1
U(0, 0): G_inv_frev = 1
U(0, 0): G_inv_star_F = 1
U(0, 0): F_inv_star_G = 1
*****************************

U(1, 1): chow = 1
U(1, 1): bbasepoly = 1
U(1, 1): chibar = 1
U(1, 1): Z_poly = 1 + x
U(1, 1): q_poly = 1
U(1, 1): Y_poly = -1 - x
U(1, 1): F_fn = 1 + x
U(1, 1): F_inv_grev = -1
U(1, 1): G_inv_frev = -1
U(1, 1): G_inv_star_F = 0
U(1, 1): F_inv_star_G = 0
*****************************

U(2, 2): chow = 1 + x
U(2, 2): bbasepoly = 1 + x
U(2, 2): chibar = -1 + x
U(2, 2): Z_poly = 1 + 2*x + x^2
U(2, 2): q_poly = 1
U(2, 2): Y_poly = 1 + 2*x + x^2
U(2, 2): F_fn = 1 + 3*x + x^2
U(2, 2): F_inv_grev = 1 - x
U(2, 2): G_inv_frev = 1 - x
U(2, 2): G_inv_star_F = 0
U(2, 2): F_inv_star_G = 0
*****************************

U(2, 3): chow = 1 + x
U(2, 3): bbasepoly = 1 + x
U(2, 3): chibar = -2 + x
U(2, 3): Z_poly = 1 + 3*x + x^2
U(2, 3): q_